# Evaluasi SLM — dataset IndoMMLU + IndoCareer
**Model**: `GANTI_NAMA_MODEL`  
**HF ID**: `GANTI_HF_ID`  
**Skema**: Zero-shot, Greedy Decoding, 4-bit Quantization  
**Benchmark**: IndoMMLU (indolem/IndoMMLU) + IndoCareer (indolem/IndoCareer,config='all') 
**Ranking**: Accuracy 50% IndoMMLU + 50% IndoCareer

## Konfiguration

In [1]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import json, os, time

MODEL_NAME = "Qwen2.5-7B"
HF_ID      = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "/kaggle/working"  # ganti ke "/content" jika pakai Colab

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


## Load Model

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(HF_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    HF_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)

model.eval()
if torch.__version__ >= "2":
    model = torch.compile(model)
print("Model loaded.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded.


In [4]:
print(next(model.parameters()).device)
print(torch.cuda.memory_reserved(0)/1e9, "GB reserved GPU 0")
print(torch.cuda.memory_reserved(1)/1e9, "GB reserved GPU 1")

cuda:0
14.766047232 GB reserved GPU 0
0.0 GB reserved GPU 1


## Fungsi Evaluasi Inti

Metode: **first-token probability** pada token A/B/C/D/E.
Sesuai dengan skema evaluasi Koto et al. (2023, 2025) yang menggunakan
probabilitas token pertama dari opsi jawaban untuk memilih prediksi.

In [5]:
CHOICE_LABELS = ["A", "B", "C", "D", "E"]

def get_choice_token_ids(tokenizer):
    ids = {}
    for label in CHOICE_LABELS:
        tokens = tokenizer.encode(label, add_special_tokens=False)
        ids[label] = tokens[-1]
    return ids

CHOICE_TOKEN_IDS = get_choice_token_ids(tokenizer)

def predict_choice_batch(prompts, valid_choices_list, batch_size=8):
    all_preds = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        batch_choices = valid_choices_list[i:i+batch_size]
        inputs = tokenizer(
            batch_prompts, return_tensors="pt",
            truncation=True, max_length=2048,
            padding=True, return_attention_mask=True
        ).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
        for j, valid_choices in enumerate(batch_choices):
            logits = outputs.logits[j, -1, :]
            scores = {c: logits[CHOICE_TOKEN_IDS[c]].item() for c in valid_choices}
            all_preds.append(max(scores, key=scores.get))
    return all_preds

## A. IndoMMLU Evaluation

Prompt sesuai paper Koto et al. (EMNLP 2023):
`Ini adalah soal [subject] untuk [level]. Pilihlah salah satu jawaban yang dianggap benar!`

In [6]:
import os, glob
import pandas as pd

os.system("git clone https://github.com/fajri91/IndoMMLU.git")

files = glob.glob("IndoMMLU/data/**/*.csv", recursive=True)
df_mmlu = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"IndoMMLU total soal: {len(df_mmlu)}")
print(df_mmlu.columns.tolist())
df_mmlu.head(2)

Cloning into 'IndoMMLU'...


IndoMMLU total soal: 14981
['id', 'sumber', 'keterangan sumber', 'kelas', 'nomor', 'subject', 'level', 'soal', 'jawaban', 'kunci', 'is_for_fewshot']


,id,sumber,keterangan sumber,kelas,nomor,subject,level,soal,jawaban,kunci,is_for_fewshot
0,0,60+ Contoh Soal PG Sejarah Kelas 10 SMA/MA dan...,PAS,10.0,1.0,Sejarah,SMA,"Peristiwa kelahiran, penerangan agung, dan kem...",A. Galungan\nB. Kuningan\nC. Paskah\nD. Waisak...,D,0
1,1,60+ Contoh Soal PG Sejarah Kelas 10 SMA/MA dan...,PAS,10.0,2.0,Sejarah,SMA,Umat Hindu memuja tiga dewa yang disebut Trimu...,A. Brahma\nB. Wisnu\nC. Syiwa\nD. Agni\nE. Baruna,A,0


In [7]:
print(df_mmlu['is_for_fewshot'].value_counts())
print()
print(df_mmlu[df_mmlu['is_for_fewshot']==1][['subject','level','kelas']].value_counts().head(20))

is_for_fewshot
0    14906
1       75
Name: count, dtype: int64

subject                  level        kelas              
Agama Hindu              SMP          7.0                    2
Bahasa Lampung           SD-SMP-SMA   6.0                    2
Bahasa Dayak Ngaju       SD-SMP-SMA   5.0                    2
Bahasa Makassar          SD-SMP-SMA   6.0                    2
Ekonomi                  SMA          12.0                   2
Biologi                  Seleksi PTN  2023-10-12 00:00:00    2
Budaya Alam Minangkabau  SD-SMP       5.0                    2
IPS                      SMP          8.0                    2
Kesenian                 SMP          7.0                    2
PPKN                     SMA          10.0                   2
Sejarah                  Seleksi PTN  PTN                    2
IPA                      SD           6                      2
Agama Islam              SMA          12.0                   1
Agama Hindu              SD           6.0                  

In [8]:
df_mmlu = df_mmlu[df_mmlu["is_for_fewshot"] == 0].reset_index(drop=True)
print(f"IndoMMLU zero-shot soal: {len(df_mmlu)}")

IndoMMLU zero-shot soal: 14906


In [9]:
import math

def build_indommlu_prompt(row):
    subject = row.get("subject", "")
    if row.get("level", "") == "Seleksi PTN":
        level = "seleksi masuk universitas"
    else:
        try:
            level = f"{math.trunc(float(row['kelas']))} {row['level']}"
        except:
            level = f"{row['kelas']} {row['level']}"
    q       = row["soal"]
    opts_raw = str(row["jawaban"]).split("\n")
    options = [o.strip() for o in opts_raw if o.strip()]
    valid_choices = [o[0] for o in options if o and o[0] in CHOICE_LABELS]
    opts_str = "\n".join(options)
    prompt = (
        f"Ini adalah soal {subject} untuk {level}. "
        f"Pilihlah salah satu jawaban yang dianggap benar!\n\n"
        f"{q}\n{opts_str}\n\nJawaban:"
    )
    return prompt, valid_choices

def run_indommlu(df):
    prompts, valid_choices_list, meta = [], [], []
    for _, row in df.iterrows():
        prompt, valid_choices = build_indommlu_prompt(row)
        if not valid_choices:
            continue
        prompts.append(prompt)
        valid_choices_list.append(valid_choices)
        meta.append({"subject": row.get("subject",""), "level": row.get("level",""), "gold": str(row["kunci"]).strip().upper()})
    
    preds = []
    for i in tqdm(range(0, len(prompts), 8), desc="IndoMMLU"):
        batch_preds = predict_choice_batch(prompts[i:i+8], valid_choices_list[i:i+8], batch_size=8)
        preds.extend(batch_preds)
    
    results = []
    for m, pred in zip(meta, preds):
        results.append({**m, "pred": pred, "correct": int(pred == m["gold"])})
    return pd.DataFrame(results)

In [10]:
df_mmlu_results = run_indommlu(df_mmlu)
acc_mmlu = df_mmlu_results["correct"].mean() * 100
print(f"\nIndoMMLU Accuracy: {acc_mmlu:.2f}%")

IndoMMLU:   0%|          | 0/1864 [00:00<?, ?it/s]

W0422 22:25:12.531000 23 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-pa


IndoMMLU Accuracy: 40.26%


In [11]:
mmlu_per_level = (
    df_mmlu_results.groupby("level")["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy", "count": "n_soal"})
    .sort_values("accuracy", ascending=False)
)
mmlu_per_level["accuracy"] = (mmlu_per_level["accuracy"] * 100).round(2)
print(mmlu_per_level)

mmlu_per_subject = (
    df_mmlu_results.groupby(["subject"])["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy", "count": "n_soal"})
    .sort_values("accuracy", ascending=False)
)
mmlu_per_subject["accuracy"] = (mmlu_per_subject["accuracy"] * 100).round(2)
print(mmlu_per_subject)

               accuracy  n_soal
level                          
Kelas II SD       66.00     150
Kelas I SD        61.81     199
Kelas IV SD       52.94     187
V SD              50.77     195
VI SD             50.76     197
SD                46.14    2640
Kelas III SD      45.13     195
VIII SMP          44.48     290
SMP               41.90    2501
SMA               40.28    3654
VII SMP           39.65     285
Kelas XI SMA      37.15     288
IX SMP            36.73     275
Kelas X SMA       35.25     295
SD-SMP            34.69     196
Seleksi PTN       31.02    2044
Kelas XII SMA     30.40     273
SD-SMP-SMA        30.33    1042
                         accuracy  n_soal
subject                                  
IPS                         57.55     596
Agama Kristen               53.03     198
Agama Islam                 52.71     700
IPA                         50.26     967
Kesenian                    48.66     598
PPKN                        46.12     696
Bahasa Indonesia        

## B. IndoCareer Evaluation

Load dengan config `'all'`. Prompt zero-shot sesuai paper Koto (NAACL 2025):
`Ini adalah soal [subject]. Pilihlah salah satu jawaban yang dianggap benar!`
Hanya gunakan split `test` (bukan `dev` yang merupakan few-shot examples).

In [12]:
raw_career = load_dataset("indolem/IndoCareer", "all", trust_remote_code=True)
df_career = pd.DataFrame(raw_career["test"])
print(f"IndoCareer total soal (test split): {len(df_career)}")
print(df_career.columns.tolist())
df_career.head(2)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'indolem/IndoCareer' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

dev.csv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/8768 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/66 [00:00<?, ? examples/s]

IndoCareer total soal (test split): 8768
['ID', 'Source', 'Subject', 'Exam Type', 'Profession Group', 'Question', 'Option A', 'Option B', 'Option C', 'Option D', 'Option E', 'Answer Key', 'is_few_shot']


,ID,Source,Subject,Exam Type,Profession Group,Question,Option A,Option B,Option C,Option D,Option E,Answer Key,is_few_shot
0,0,https://www.scribd.com/document/502335478/SOAL...,Tourism,Vocational School,Tourism and Hospitality,Tahap pramuwisata yang awal adalah…,Pramuwisata Muda,Pramuwisata Utama,Pramuwisata Lokal,Pramuwisata Madya,Pramuwisata Profesional,A,0
1,1,https://www.scribd.com/document/502335478/SOAL...,Tourism,Vocational School,Tourism and Hospitality,"“Well Ladies and Gentleman, welcome to Sri Aji...",General information,On site/object explanation,General instruction,On spot explanation,General manager,B,0


In [13]:
print(df_career['is_few_shot'].value_counts())
print()
print(df_career[df_career['is_few_shot']==1][['Subject','Exam Type','Profession Group']].value_counts().head(20))

is_few_shot
0    8768
Name: count, dtype: int64

Series([], Name: count, dtype: int64)


In [14]:
df_career = df_career[df_career["is_few_shot"] == 0].reset_index(drop=True)
print(f"IndoMMLU zero-shot soal: {len(df_career)}")

IndoMMLU zero-shot soal: 8768


In [15]:
def build_indocareer_prompt(row):
    subject = row.get("Subject", "")
    q       = row["Question"]
    options = []
    for label in CHOICE_LABELS:
        key = f"Option {label}"
        if key in row and pd.notna(row[key]) and str(row[key]).strip():
            options.append(f"{label}. {row[key]}")
    valid_choices = [o[0] for o in options]
    opts_str = "\n".join(options)
    prompt = (
        f"Ini adalah soal {subject}. "
        f"Pilihlah salah satu jawaban yang dianggap benar!\n\n"
        f"{q}\n{opts_str}\n\nJawaban:"
    )
    return prompt, valid_choices

def run_indocareer(df):
    prompts, valid_choices_list, meta = [], [], []
    for _, row in df.iterrows():
        prompt, valid_choices = build_indocareer_prompt(row)
        if not valid_choices:
            continue
        prompts.append(prompt)
        valid_choices_list.append(valid_choices)
        meta.append({
            "subject": row.get("Subject", ""),
            "profession_group": row.get("Profession Group", ""),
            "gold": str(row["Answer Key"]).strip().upper()
        })
    
    preds = []
    for i in tqdm(range(0, len(prompts), 8), desc="IndoCareer"):
        batch_preds = predict_choice_batch(prompts[i:i+8], valid_choices_list[i:i+8], batch_size=8)
        preds.extend(batch_preds)
    
    results = []
    for m, pred in zip(meta, preds):
        results.append({**m, "pred": pred, "correct": int(pred == m["gold"])})
    return pd.DataFrame(results)

In [16]:
df_career_results = run_indocareer(df_career)
acc_career = df_career_results["correct"].mean() * 100
print(f"\nIndoCareer Accuracy: {acc_career:.2f}%")

IndoCareer:   0%|          | 0/1096 [00:00<?, ?it/s]


IndoCareer Accuracy: 41.71%


In [17]:
career_per_subject = (
    df_career_results.groupby("subject")["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy", "count": "n_soal"})
    .sort_values("accuracy", ascending=False)
)
career_per_subject["accuracy"] = (career_per_subject["accuracy"] * 100).round(2)
print(career_per_subject)

career_per_group = (
    df_career_results.groupby("profession_group")["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy", "count": "n_soal"})
    .sort_values("accuracy", ascending=False)
)
career_per_group["accuracy"] = (career_per_group["accuracy"] * 100).round(2)
print(career_per_group)

                                              accuracy  n_soal
subject                                                       
Graphic Design                                   55.24     420
Sharia Life Insurance                            54.95     555
Life Insurance                                   52.85     473
Clinical Psychology                              51.09      92
Broadcasting                                     49.88     419
Cullinary Art                                    47.77     291
Office Administration                            47.23     343
Risk Management                                  45.55     191
Police                                           45.13     277
Teacher Competency Test                          41.68     535
Tourism                                          41.55     219
Hospitality                                      40.11     364
Certified Public Accountant                      39.09     660
Advocate                                         38.78 

## Final Score & Simpan Hasil

In [18]:
final_score = 0.5 * acc_mmlu + 0.5 * acc_career
summary = {
    "model_name":        MODEL_NAME,
    "hf_id":             HF_ID,
    "indommlu_acc":      round(acc_mmlu, 4),
    "indocareer_acc":    round(acc_career, 4),
    "final_score_50_50": round(final_score, 4),
    "indommlu_n":        len(df_mmlu_results),
    "indocareer_n":      len(df_career_results),
}

print("="*50)
print(f"  Model        : {MODEL_NAME}")
print(f"  IndoMMLU     : {acc_mmlu:.2f}%  (n={summary['indommlu_n']})")
print(f"  IndoCareer   : {acc_career:.2f}%  (n={summary['indocareer_n']})")
print(f"  FINAL SCORE  : {final_score:.2f}%")
print("="*50)

safe_name = MODEL_NAME.replace("/", "-").replace(" ", "_")

with open(f"{OUTPUT_DIR}/summary_{safe_name}.json", "w") as f:
    json.dump(summary, f, indent=2)

df_mmlu_results.to_csv(f"{OUTPUT_DIR}/detail_indommlu_{safe_name}.csv", index=False)
df_career_results.to_csv(f"{OUTPUT_DIR}/detail_indocareer_{safe_name}.csv", index=False)

mmlu_per_subject["model"] = MODEL_NAME
mmlu_per_subject.to_csv(f"{OUTPUT_DIR}/breakdown_mmlu_subject_{safe_name}.csv")

mmlu_per_level["model"] = MODEL_NAME
mmlu_per_level.to_csv(f"{OUTPUT_DIR}/breakdown_mmlu_level_{safe_name}.csv")

career_per_subject["model"] = MODEL_NAME
career_per_subject.to_csv(f"{OUTPUT_DIR}/breakdown_career_subject_{safe_name}.csv")

career_per_group["model"] = MODEL_NAME
career_per_group.to_csv(f"{OUTPUT_DIR}/breakdown_career_group_{safe_name}.csv")

print(f"\nHasil disimpan di: {OUTPUT_DIR}")

  Model        : Qwen2.5-7B
  IndoMMLU     : 40.26%  (n=14906)
  IndoCareer   : 41.71%  (n=8768)
  FINAL SCORE  : 40.98%

Hasil disimpan di: /kaggle/working
